In [ ]:






import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, ttest_1samp
pd.set_option("display.float_format", "{:.4f}".format)


def transform_event_time_for_matching(
    df,
    cohort_col="cohort",
    event_col="event_time",
    target_months=[1, 2, 3, 11, 12],
    pre_start=-2,
    new_event_col="new_event_time"
):
    df = df.copy()

    if pd.api.types.is_period_dtype(df[cohort_col]):
        cohort_date = df[cohort_col].dt.to_timestamp()
    else:
        cohort_date = pd.to_datetime(df[cohort_col], errors="coerce")

    df["cohort_month"] = cohort_date.dt.month

    df["cohort_event_month"] = (
        (df["cohort_month"] + df[event_col] - 1) % 12
    ) + 1

    df["target_month_flag"] = (
        df["cohort_event_month"].isin(target_months).astype(int)
    )

    df[new_event_col] = pd.NA

    def reindex(g):
        g = g.copy()

        # Pre-period: target months before first effective exposure
        pre_times = (
            g.loc[
                (g[event_col] < 0) &
                (g["target_month_flag"] == 1),
                event_col
            ]
            .drop_duplicates()
            .sort_values(ascending=False)
            .tolist()
        )

        pre_map = {
            old_t: new_t
            for old_t, new_t in zip(
                pre_times,
                range(pre_start, pre_start - len(pre_times), -1)
            )
        }

        # Post-period: first effective high-price month = event time 0
        post_times = (
            g.loc[
                (g[event_col] >= 0) &
                (g["target_month_flag"] == 1),
                event_col
            ]
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        post_map = {
            old_t: new_t
            for old_t, new_t in zip(
                post_times,
                range(0, len(post_times))
            )
        }

        time_map = {**pre_map, **post_map}

        g.loc[g[event_col].isin(time_map.keys()), new_event_col] = (
            g.loc[g[event_col].isin(time_map.keys()), event_col].map(time_map)
        )

        return g

    df = df.groupby(cohort_col, group_keys=False).apply(reindex)
    df[new_event_col] = df[new_event_col].astype("Int64")

    return df


# Build matched panel
def build_matched_panel(matches, month_result):

    matches = matches.copy()
    month_result = month_result.copy()

    matches["cohort"] = pd.to_datetime(matches["adoption_month"]).dt.to_period("M")

    treated_map = matches[["treated_id", "cohort"]].drop_duplicates().copy()
    treated_map.columns = ["aID", "cohort"]
    treated_map["treatment"] = 1

    control_map = matches[["control_id", "cohort"]].copy()
    control_map.columns = ["aID", "cohort"]
    control_map["treatment"] = 0

    match_map = pd.concat([treated_map, control_map], axis=0, ignore_index=True)

    df = month_result.merge(match_map, on="aID", how="inner")

    df["TIDPUNKT"] = pd.to_datetime(df["TIDPUNKT"]).dt.to_period("M")
    df["event_time"] = (df["TIDPUNKT"] - df["cohort"]).apply(lambda x: x.n)
    df["top3_mean_consumption"] = pd.to_numeric(df["top3_mean_consumption"], errors="coerce")

    return df


# Monthly treatment effect + p-value
def compute_effect(df, outcome_col="top3_mean_consumption", event_col="event_time"):

    results = []

    for t in sorted(df[event_col].dropna().unique()):
        d = df[df[event_col] == t]

        treated = d[d["treatment"] == 1][outcome_col]
        control = d[d["treatment"] == 0][outcome_col]

        if len(treated) < 2 or len(control) < 2:
            continue

        mean_t = treated.mean()
        mean_c = control.mean()

        var_t = treated.var(ddof=1)
        var_c = control.var(ddof=1)

        n_t = len(treated)
        n_c = len(control)

        effect = mean_t - mean_c
        se = np.sqrt(var_t / n_t + var_c / n_c)

        t_stat = effect / se if se > 0 else np.nan
        p_value = 2 * (1 - norm.cdf(abs(t_stat))) if se > 0 else np.nan

        results.append({
            event_col: t,
            "mean_treated": mean_t,
            "mean_control": mean_c,
            "effect": effect,
            "se": se,
            "t_stat": t_stat,
            "p_value": p_value,
            "n_treated": n_t,
            "n_control": n_c
        })

    return pd.DataFrame(results)


# Cohort-level monthly effect + p-value
def compute_effect_by_cohort(
    df,
    outcome_col="top3_mean_consumption",
    event_col="event_time"
):

    results = []

    for c in sorted(df["cohort"].dropna().unique()):
        d_cohort = df[df["cohort"] == c]

        for t in sorted(d_cohort[event_col].dropna().unique()):
            d = d_cohort[d_cohort[event_col] == t]

            treated = d[d["treatment"] == 1][outcome_col]
            control = d[d["treatment"] == 0][outcome_col]

            if len(treated) < 2 or len(control) < 2:
                continue

            effect = treated.mean() - control.mean()
            se = np.sqrt(
                treated.var(ddof=1) / len(treated) +
                control.var(ddof=1) / len(control)
            )

            t_stat = effect / se if se > 0 else np.nan
            p_value = 2 * (1 - norm.cdf(abs(t_stat))) if se > 0 else np.nan

            results.append({
                "cohort": c,
                event_col: t,
                "effect": effect,
                "se": se,
                "t_stat": t_stat,
                "p_value": p_value,
                "n_treated": len(treated),
                "n_control": len(control)
            })

    return pd.DataFrame(results)


# Confidence interval
def add_confidence_interval(effect_df, alpha=0.05):

    effect_df = effect_df.copy()

    z = norm.ppf(1 - alpha / 2)

    effect_df["ci_low"] = effect_df["effect"] - z * effect_df["se"]
    effect_df["ci_high"] = effect_df["effect"] + z * effect_df["se"]

    return effect_df


# Pre-trend test
def pretrend_test(effect_df, event_col="event_time", pre_start=-12, pre_end=-1):

    pre = effect_df[
        (effect_df[event_col] >= pre_start) &
        (effect_df[event_col] <= pre_end)
    ].copy()

    pre = pre.dropna(subset=["effect"])

    if len(pre) < 2:
        return {
            "pre_start": pre_start,
            "pre_end": pre_end,
            "n_periods": len(pre),
            "mean_pre_effect": np.nan,
            "t_stat": np.nan,
            "p_value": np.nan
        }

    test = ttest_1samp(pre["effect"], popmean=0)

    return {
        "pre_start": pre_start,
        "pre_end": pre_end,
        "n_periods": len(pre),
        "mean_pre_effect": pre["effect"].mean(),
        "t_stat": test.statistic,
        "p_value": test.pvalue
    }




# Dynamic effect plot
def plot_dynamic_effect(
    effect_df,
    event_col="event_time",
    ylim=None
):

    d = effect_df.sort_values(event_col)

    plt.figure(figsize=(7, 5))
    plt.plot(d[event_col], d["effect"], marker="o")

    plt.fill_between(
        d[event_col],
        d["ci_low"],
        d["ci_high"],
        alpha=0.2
    )

    plt.axhline(0, linestyle="--")
    plt.axvline(0, linestyle="--")

    if ylim is not None:
        plt.ylim(ylim)

    plt.xlabel("Event Time Relative to Tariff Exposure")
    plt.ylabel("Treatment Effect")
    plt.title("Dynamic Treatment Effect")

    xmin = int(d[event_col].min())
    xmax = int(d[event_col].max())

    plt.xticks(
        np.arange(
            2 * (xmin // 2),
            xmax + 1,
            2
        )
    )

    plt.show()


def plot_dynamic_by_cohort(
    effect_df,
    event_col="event_time",
    ylim=None
):

    for c in effect_df["cohort"].unique():

        d = effect_df[
            effect_df["cohort"] == c
        ].sort_values(event_col)

        plt.figure(figsize=(7, 5))

        plt.plot(
            d[event_col],
            d["effect"],
            marker="o"
        )

        plt.fill_between(
            d[event_col],
            d["ci_low"],
            d["ci_high"],
            alpha=0.2
        )

        plt.axhline(0, linestyle="--")
        plt.axvline(0, linestyle="--")

        if ylim is not None:
            plt.ylim(ylim)

        plt.title(f"Cohort = {c}")
        plt.xlabel("Event Time Relative to Tariff Exposure")
        plt.ylabel("Treatment Effect")

        xmin = int(d[event_col].min())
        xmax = int(d[event_col].max())

        plt.xticks(
            np.arange(
                2 * (xmin // 2),
                xmax + 1,
                2
            )
        )

        plt.show()


def plot_all_cohorts(
    effect_df,
    event_col="event_time",
    ylim=None
):

    plt.figure(figsize=(7, 5))

    for c in effect_df["cohort"].unique():

        d = effect_df[
            effect_df["cohort"] == c
        ].sort_values(event_col)

        plt.plot(
            d[event_col],
            d["effect"],
            alpha=0.5,
            label=str(c)
        )

    plt.axhline(0, linestyle="--")
    plt.axvline(0, linestyle="--")

    if ylim is not None:
        plt.ylim(ylim)

    plt.xlabel("Event Time Relative to Tariff Exposure")
    plt.ylabel("Treatment Effect")
    plt.title("Dynamic Effect by Cohort")

    plt.legend(bbox_to_anchor=(1.05, 1))
    plt.show()


# Average treatment effect ATT + p-value
def compute_average_treatment_effect(
    effect_df,
    event_col="event_time",
    post_period_only=True,
    max_post_periods=None
):

    df = effect_df.copy()

    if post_period_only:
        df = df[df[event_col] >= 0]

    if max_post_periods is not None:
        df = df[df[event_col] < max_post_periods]

    df = df.dropna(subset=["effect"])

    if len(df) < 2:
        return {
            "ATT": np.nan,
            "SE": np.nan,
            "t_stat": np.nan,
            "p_value": np.nan,
            "CI_low": np.nan,
            "CI_high": np.nan,
            "n_periods": len(df)
        }

    att = df["effect"].mean()
    se = df["effect"].std(ddof=1) / np.sqrt(len(df))

    return {
        "ATT": att,
        "SE": se,
        "t_stat": att / se if se > 0 else np.nan,
        "p_value": 2 * (1 - norm.cdf(abs(att / se))) if se > 0 else np.nan,
        "CI_low": att - 1.96 * se,
        "CI_high": att + 1.96 * se,
        "n_periods": len(df)
    }




def compute_att_by_cohort(
    effect_df,
    event_col="event_time",
    post_period_only=True,
    max_post_periods=None
):

    results = []

    for c in effect_df["cohort"].unique():

        d = effect_df[effect_df["cohort"] == c].copy()

        if post_period_only:
            d = d[d[event_col] >= 0]

        if max_post_periods is not None:
            d = d[d[event_col] < max_post_periods]

        d = d.dropna(subset=["effect"])

        if len(d) < 2:
            continue

        att = d["effect"].mean()
        se = d["effect"].std(ddof=1) / np.sqrt(len(d))

        results.append({
            "cohort": c,
            "ATT": att,
            "SE": se,
            "t_stat": att / se if se > 0 else np.nan,
            "p_value": 2 * (1 - norm.cdf(abs(att / se))) if se > 0 else np.nan,
            "CI_low": att - 1.96 * se,
            "CI_high": att + 1.96 * se,
            "n_periods": len(d)
        })

    return pd.DataFrame(results)


def assign_cohort_group(
    df,
    cohort_col="cohort"
):

    df = df.copy()

    # cohort month
    cohort_month = (
        df[cohort_col]
        .dt.month
    )

    # cohort year
    cohort_year = (
        df[cohort_col]
        .dt.year
    )

    # 4-10 月
    # mask_4_10 = cohort_month.between(4, 10)

    # df["cohort_group"] = np.where(
    #     mask_4_10,
    #     cohort_year.astype(str) + "-04 to " + cohort_year.astype(str) + "-10",
    #     cohort_year.astype(str) + "-" + cohort_month.astype(str).str.zfill(2)
    # )
    # 4-10 月
    # 4-11 月
    mask_4_11 = cohort_month.between(4, 11)

    df["cohort_group"] = np.where(
        mask_4_11,
        cohort_year.astype(str) + "-04 to " + cohort_year.astype(str) + "-11",
        cohort_year.astype(str) + "-" + cohort_month.astype(str).str.zfill(2)
    )

    return df


def compute_effect_by_cohort_group(
    df,
    outcome_col="top3_mean_consumption",
    event_col="new_event_time",
    cohort_group_col="cohort_group"
):

    results = []

    for g in sorted(df[cohort_group_col].dropna().unique()):

        d_group = df[df[cohort_group_col] == g]

        for t in sorted(d_group[event_col].dropna().unique()):

            d = d_group[d_group[event_col] == t]

            treated = d[d["treatment"] == 1][outcome_col]
            control = d[d["treatment"] == 0][outcome_col]

            if len(treated) < 2 or len(control) < 2:
                continue

            mean_t = treated.mean()
            mean_c = control.mean()

            var_t = treated.var(ddof=1)
            var_c = control.var(ddof=1)

            n_t = len(treated)
            n_c = len(control)

            effect = mean_t - mean_c

            se = np.sqrt(
                var_t / n_t +
                var_c / n_c
            )

            t_stat = effect / se if se > 0 else np.nan

            p_value = (
                2 * (1 - norm.cdf(abs(t_stat)))
                if se > 0 else np.nan
            )

            results.append({
                event_col: t,
                "effect": effect,
                "se": se,
                "p_value": p_value,
                "ci_low": effect - 1.96 * se,
                "ci_high": effect + 1.96 * se,
                "n_control": n_c,
                "n_treated": n_t,
                cohort_group_col: g
            })

    return pd.DataFrame(results)


def plot_dynamic_by_cohort_group(
    effect_df,
    event_col="new_event_time",
    cohort_group_col="cohort_group",
    ylim=None
):

    for g in effect_df[cohort_group_col].unique():

        d = (
            effect_df[
                effect_df[cohort_group_col] == g
            ]
            .sort_values(event_col)
        )

        plt.figure(figsize=(7, 5))

        plt.plot(
            d[event_col],
            d["effect"],
            marker="o"
        )

        plt.fill_between(
            d[event_col],
            d["ci_low"],
            d["ci_high"],
            alpha=0.2
        )

        plt.axhline(0, linestyle="--")
        plt.axvline(0, linestyle="--")

        if ylim is not None:
            plt.ylim(ylim)

        plt.title(f"Cohort Group = {g}")

        plt.xlabel("Event Time Relative to Tariff Exposure")
        plt.ylabel("Treatment Effect")

        xmin = int(d[event_col].min())
        xmax = int(d[event_col].max())

        plt.xticks(
            np.arange(
                2 * (xmin // 2),
                xmax + 1,
                2
            )
        )

        plt.show()


def run_full_analysis(
    df,
    outcome_col="top3_mean_consumption",
    pre_start=-12,
    pre_end=-1,
    event_col="event_time",
    use_transformed_event_time=True,
    target_months=[1, 2, 3, 11, 12],
    transformed_event_col="new_event_time",
    transformed_pre_start=-1,
    ylim=None
):

    df = df.copy()

    if use_transformed_event_time:
        df = transform_event_time_for_matching(
            df,
            cohort_col="cohort",
            event_col=event_col,
            target_months=target_months,
            pre_start=transformed_pre_start,
            new_event_col=transformed_event_col
        )

        print("\n===== TRANSFORMED EVENT TIME DATA =====")
        print(
            df[
                [
                    "aID",
                    "cohort",
                    "TIDPUNKT",
                    "treatment",
                    event_col,
                    "cohort_event_month",
                    "target_month_flag",
                    transformed_event_col
                ]
            ]
            .sort_values(["cohort", "aID", event_col])
            .head(100)
        )

        event_col = transformed_event_col
        df = df.dropna(subset=[event_col])

    print("===== OVERALL EFFECT =====")

    overall_df = compute_effect(df, outcome_col, event_col=event_col)
    overall_df = add_confidence_interval(overall_df)

    print("\nOverall dynamic effect:")
    print(overall_df.round(4))

    plot_dynamic_effect(overall_df, event_col=event_col, ylim=ylim)

    print("\n===== PRE-TREND TEST =====")
    pretrend = pretrend_test(
        overall_df,
        event_col=event_col,
        pre_start=pre_start,
        pre_end=pre_end
    )
    print(pretrend)

    print("\n===== OVERALL ATT: ALL POST PERIODS =====")
    att_all = compute_average_treatment_effect(
        overall_df,
        event_col=event_col,
        post_period_only=True,
        max_post_periods=None
    )
    print(att_all)

    print("\n===== OVERALL ATT: FIRST 3 POST PERIODS =====")
    att_3m = compute_average_treatment_effect(
        overall_df,
        event_col=event_col,
        post_period_only=True,
        max_post_periods=3
    )
    print(att_3m)

    print("\n===== COHORT DYNAMIC EFFECT =====")

    cohort_df = compute_effect_by_cohort(df, outcome_col, event_col=event_col)
    cohort_df = add_confidence_interval(cohort_df)

    print("\nCohort dynamic effect:")
    print(cohort_df.round(4))

    plot_dynamic_by_cohort(cohort_df, event_col=event_col, ylim=ylim)
    plot_all_cohorts(cohort_df, event_col=event_col, ylim=ylim)

    print("\n===== ATT BY COHORT: ALL POST PERIODS =====")
    att_cohort_all_df = compute_att_by_cohort(
        cohort_df,
        event_col=event_col,
        post_period_only=True,
        max_post_periods=None
    )
    print(att_cohort_all_df.round(4))

    print("\n===== ATT BY COHORT: FIRST 3 POST PERIODS =====")
    att_cohort_3m_df = compute_att_by_cohort(
        cohort_df,
        event_col=event_col,
        post_period_only=True,
        max_post_periods=3
    )
    print(att_cohort_3m_df.round(4))


    df = assign_cohort_group(df)

    group_df = compute_effect_by_cohort_group(
        df,
        event_col="new_event_time"
    )

    print(group_df)

    plot_dynamic_by_cohort_group(
        group_df,
        event_col="new_event_time",
        ylim=ylim
    )

    return {
        "overall": overall_df,
        "cohort": cohort_df,
        "group": group_df,
        "att_all": att_all,
        "att_3m": att_3m,
        "att_by_cohort_all": att_cohort_all_df,
        "att_by_cohort_3m": att_cohort_3m_df,
        "pretrend": pretrend,
        "analysis_event_col": event_col,
        "analysis_df": df
    }


def save_results(results, save_path):

    os.makedirs(save_path, exist_ok=True)

    if "overall" in results:
        results["overall"].to_csv(
            os.path.join(save_path, "overall_dynamic.csv"),
            index=False
        )

    if "cohort" in results:
        results["cohort"].to_csv(
            os.path.join(save_path, "cohort_dynamic.csv"),
            index=False
        )

    if "att" in results:
        pd.DataFrame([results["att"]]).to_csv(
            os.path.join(save_path, "att_overall.csv"),
            index=False
        )

    if "att_by_cohort" in results:
        results["att_by_cohort"].to_csv(
            os.path.join(save_path, "att_by_cohort.csv"),
            index=False
        )

    if "pretrend" in results:
        pd.DataFrame([results["pretrend"]]).to_csv(
            os.path.join(save_path, "pretrend_test.csv"),
            index=False
        )

    if "att_all" in results:
        pd.DataFrame([results["att_all"]]).to_csv(
            os.path.join(save_path, "att_overall_all_post.csv"),
            index=False
        )

    if "att_3m" in results:
        pd.DataFrame([results["att_3m"]]).to_csv(
            os.path.join(save_path, "att_overall_first_3m.csv"),
            index=False
        )

    if "att_by_cohort_all" in results:
        results["att_by_cohort_all"].to_csv(
            os.path.join(save_path, "att_by_cohort_all_post.csv"),
            index=False
        )

    if "att_by_cohort_3m" in results:
        results["att_by_cohort_3m"].to_csv(
            os.path.join(save_path, "att_by_cohort_first_3m.csv"),
            index=False
        )

    if "group" in results:
        results["group"].to_csv(
            os.path.join(save_path, "group_dynamic.csv"),
            index=False
        )
        
    print(f"✅ Results saved to: {save_path}")
     
    
     